In [41]:
from datasets import load_dataset, Dataset
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix, roc_auc_score, matthews_corrcoef, classification_report

In [39]:
ds = load_dataset("cybersectony/PhishingEmailDetectionv2.0", split="train")  
print(ds)
print(ds.column_names)
print(ds[0])

Dataset({
    features: ['content', 'label'],
    num_rows: 120000
})
['content', 'label']
{'content': 'https://www.galvnews.com', 'label': 2}


In [8]:
email_ds = ds.filter(lambda x: x["label"] in (0, 1))  

print("email rows:", email_ds.num_rows)
print(email_ds[0])

email rows: 13493
{'content': 'empty', 'label': 1}


In [10]:
def is_valid_email(x):
    text = x["content"]
    if text is None:
        return False
    text = str(text).strip().lower()
    if text == "" or text == "empty":
        return False
    return True

In [12]:
email_ds_clean = email_ds.filter(is_valid_email)
print("clean email rows:", email_ds_clean.num_rows)


clean email rows: 13074


In [14]:
email_6k = email_ds_clean.shuffle(seed=42).select(range(6000))
email_6k = email_6k.map(lambda x: {"y": 1 if x["label"] == 1 else 0})  # phishing=1, legit=0 [web:25]

print(email_6k.num_rows, email_6k.column_names)

6000 ['content', 'label', 'y']


In [16]:
from collections import Counter
Counter(email_6k["y"])

Counter({0: 3043, 1: 2957})

In [18]:
X = email_6k["content"]
y = email_6k["y"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [20]:
lr_clf = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2)),
    ("clf", LogisticRegression(max_iter=2000, class_weight="balanced"))
])
lr_clf.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [22]:
dt_clf = Pipeline([
    ("tfidf", TfidfVectorizer(lowercase=True, ngram_range=(1,2), min_df=2)),
    ("clf", DecisionTreeClassifier(random_state=42, class_weight="balanced"))
])
dt_clf.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('tfidf', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'
,"strip_accents strip_accents: {'ascii', 'unicode'} or callable, default=NoneRemove accents and perform other character normalizationduring the preprocessing step.'ascii' is a fast method that only works on characters that havea direct ASCII mapping.'unicode' is a slightly slower method that works on any characters.None (default) means no character normalization is performed.Both 'ascii' and 'unicode' use NFKD normalization from:func:`unicodedata.normalize`.",None
,"lowercase lowercase: bool, default=TrueConvert all characters to lowercase before tokenizing.",True
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (string transformation) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",None
,"tokenizer tokenizer: callable, default=NoneOverride the string tokenization step while preserving thepreprocessing and n-grams generation steps.Only applies if ``analyzer == 'word'``.",None


In [26]:
def evaluate(model, X_test, y_test):
    y_pred = model.predict(X_test)
    y_score = model.predict_proba(X_test)[:, 1]  # вероятность phishing

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()  # [web:179]

    tpr = tp / (tp + fn) if (tp + fn) else 0.0
    fnr = fn / (tp + fn) if (tp + fn) else 0.0
    fpr = fp / (fp + tn) if (fp + tn) else 0.0
    specificity = tn / (tn + fp) if (tn + fp) else 0.0

    return {
        "TP": tp, "TN": tn, "FP": fp, "FN": fn,
        "TPR": tpr,
        "FNR": fnr,
        "FPR": fpr,
        "Specificity": specificity,
        "ROC-AUC": roc_auc_score(y_test, y_score),
        "MCC": matthews_corrcoef(y_test, y_pred),
    }

lr_metrics = evaluate(lr_clf, X_test, y_test)
dt_metrics = evaluate(dt_clf, X_test, y_test)

results = pd.DataFrame([lr_metrics, dt_metrics], index=["LogReg", "DecisionTree"])
results


,TP,TN,FP,FN,TPR,FNR,FPR,Specificity,ROC-AUC,MCC
LogReg,570,588,21,21,0.964467,0.035533,0.034483,0.965517,0.991356,0.929984
DecisionTree,535,534,75,56,0.905245,0.094755,0.123153,0.876847,0.891046,0.782113


In [32]:
y_pred_lr = lr_clf.predict(X_test)
print("=== Logistic Regression: classification report ===")
print(classification_report(
    y_test, y_pred_lr,
    target_names=["legitimate_email (0)", "phishing_email (1)"],
    digits=4,
    zero_division=0
))  


y_pred_dt = dt_clf.predict(X_test)
print("=== Decision Tree: classification report ===")
print(classification_report(
    y_test, y_pred_dt,
    target_names=["legitimate_email (0)", "phishing_email (1)"],
    digits=4,
    zero_division=0
)) 

=== Logistic Regression: classification report ===
                      precision    recall  f1-score   support

legitimate_email (0)     0.9655    0.9655    0.9655       609
  phishing_email (1)     0.9645    0.9645    0.9645       591

            accuracy                         0.9650      1200
           macro avg     0.9650    0.9650    0.9650      1200
        weighted avg     0.9650    0.9650    0.9650      1200

=== Decision Tree: classification report ===
                      precision    recall  f1-score   support

legitimate_email (0)     0.9051    0.8768    0.8907       609
  phishing_email (1)     0.8770    0.9052    0.8909       591

            accuracy                         0.8908      1200
           macro avg     0.8911    0.8910    0.8908      1200
        weighted avg     0.8913    0.8908    0.8908      1200



# С моделями глубокого обучения

In [34]:
pip install -U transformers accelerate evaluate torch scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 9.4 MB/s  0:00:01m0:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 10.4 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.4/79.4 MB 9.9 MB/s  0:00:086m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 8.7 MB/s  0:00:00m eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.3/6.3 MB 8.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 536.2/536.2 kB 7.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10/10 [evaluate]/10 [transformers]
Note: you may need to restart the kernel to use updated packages.


In [43]:
train_hf = Dataset.from_dict({"text": list(X_train), "labels": list(y_train)})
test_hf  = Dataset.from_dict({"text": list(X_test),  "labels": list(y_test)})

In [45]:
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments, Trainer
)

def finetune_model(model_name, train_hf, test_hf, max_len=256, epochs=2):
    tokenizer = AutoTokenizer.from_pretrained(model_name)

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, max_length=max_len)

    train_tok = train_hf.map(tok, batched=True)
    test_tok  = test_hf.map(tok, batched=True)

    data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2) 

    args = TrainingArguments(
        output_dir=f"./phish_{model_name.replace('/', '_')}",
        evaluation_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",
        num_train_epochs=epochs,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        learning_rate=2e-5,
        weight_decay=0.01,
        report_to="none"
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=test_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics
    )  
    
    trainer.train()
    return trainer


In [47]:
models = [
    "distilbert-base-uncased",
    "roberta-base",
    "microsoft/deberta-v3-base",
]

trainers = {}
for m in models:
    trainers[m] = finetune_model(m, train_hf, test_hf, max_len=256, epochs=2)
    print(m, trainers[m].evaluate())

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 1261.03it/s, Materializing param=distilbert.transformer.layer.5.sa_layer_norm.weight]   
DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
classifier.weight       | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'